# PAML 正向选择分析教程 (基于 Python)
本教程根据 `positive_selection_pipeline.py` 拆解了整个 PAML 正向选择分析的流程。我们将一步步执行数据验证、序列比对、反向翻译、建树以及使用 Codeml 分析正向选择位点。

## 环境准备
在开始之前，请确保当前环境中已安装 `biopython`、`scipy` 以及外部命令行工具 `mafft`、`fasttree` 和 `paml`。

In [1]:
import os
import subprocess
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from scipy.stats import chi2
import re
import shutil
import shlex

def run_command(cmd, log_error=True):
    try:
        subprocess.run(cmd, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(f"Command executed successfully: {cmd}")
    except subprocess.CalledProcessError as e:
        if log_error:
            print(f"Command failed: {cmd}")
            print(f"Error message: {e.stderr.decode('utf-8')}")
        raise

# 创建必要的输出文件夹
out_dir = "tutorial_output"
for sub in ["alignments", "trees", "paml_logs", "reports"]:
    os.makedirs(os.path.join(out_dir, sub), exist_ok=True)
    
print("输出目录创建完毕。")

输出目录创建完毕。


## 1. 序列验证与蛋白质翻译
输入必须是同源的编码序列 (CDS)。为了防止直接基于核苷酸比对导致移码，标准的做法是先将核苷酸序列翻译成氨基酸。
在此步骤中，我们：
1. 检查序列长度是否为 3 的倍数。
2. 将其翻译为蛋白质序列（默认使用细菌密码子表 Table 11）。
3. 移除序列末尾的终止密码子。

In [2]:
def validate_and_translate(fasta_in, prefix, out_dir):
    prot_fasta = os.path.join(out_dir, "alignments", f"{prefix}.prot.fasta")
    clean_nuc_fasta = os.path.join(out_dir, "alignments", f"{prefix}.clean_nuc.fasta")
    
    valid_records = []
    prot_records = []
    
    with open(fasta_in, "r") as f:
        records = list(SeqIO.parse(f, "fasta"))
        
    for rec in records:
        seq_str = str(rec.seq).upper()
        if len(seq_str) % 3 != 0:
            print(f"Warning: {rec.id} length not multiple of 3. Truncating.")
            seq_str = seq_str[:-(len(seq_str)%3)]
            
        my_seq = Seq(seq_str)
        try:
            prot_seq = my_seq.translate(table=11)
        except Exception as e:
            continue
            
        if "*" in prot_seq[:-1]:
            print(f"Warning: Internal stop codon in {rec.id}. Skipping.")
            continue
            
        if prot_seq.endswith("*"):
            prot_seq = prot_seq[:-1]
            seq_str = seq_str[:-3]
            
        clean_rec = SeqRecord(Seq(seq_str), id=rec.id, description="")
        prot_rec = SeqRecord(prot_seq, id=rec.id, description="")
        valid_records.append(clean_rec)
        prot_records.append(prot_rec)
        
    SeqIO.write(valid_records, clean_nuc_fasta, "fasta")
    SeqIO.write(prot_records, prot_fasta, "fasta")
    print(f"翻译完成：保留了 {len(valid_records)} 条有效序列。")
    return clean_nuc_fasta, prot_fasta

# 为本教程创建一个测试输入文件
test_fasta = "test_gene.fasta"
with open(test_fasta, "w") as f:
    f.write(">seq1\nATGCGTATCGATCGTACGATCGTACGTAGCTAG\n")
    f.write(">seq2\nATGCGTATCGATCGTACGATCGTACGTAGCTAG\n")
    f.write(">seq3\nATGCGTATCGATCGTACGATCGTACGTAGCTAA\n")

prefix = "test_gene"
clean_nuc, prot_fasta = validate_and_translate(test_fasta, prefix, out_dir)

翻译完成：保留了 3 条有效序列。


## 2. 氨基酸比对与反向翻译 (Back-translation)
接下来，我们调用 `MAFFT` 对刚才得到的氨基酸序列进行多序列比对 (MSA)。
然后，我们将比对好的氨基酸序列重新映射回原始的核苷酸序列，得到 **密码子对齐的核苷酸比对文件 (Codon Alignment)**。这对于 PAML 寻找正向选择位点是必需的。

In [3]:
def align_proteins_and_back_translate(prot_fasta, clean_nuc_fasta, prefix, out_dir):
    aln_fasta = os.path.join(out_dir, "alignments", f"{prefix}.prot.aln.fasta")
    cmd = f"mafft --auto {shlex.quote(prot_fasta)} > {shlex.quote(aln_fasta)}"
    run_command(cmd)
    
    # Back-translation
    codon_aln_fasta = os.path.join(out_dir, "alignments", f"{prefix}.codon.aln.fasta")
    codon_aln_phylip = os.path.join(out_dir, "alignments", f"{prefix}.codon.aln.phy")
    
    nuc_dict = SeqIO.to_dict(SeqIO.parse(clean_nuc_fasta, "fasta"))
    prot_aln_records = list(SeqIO.parse(aln_fasta, "fasta"))
    
    codon_aln_records = []
    for prot_rec in prot_aln_records:
        nuc_seq = str(nuc_dict[prot_rec.id].seq)
        prot_aln_seq = str(prot_rec.seq)
        
        codon_aln = []
        nuc_idx = 0
        for aa in prot_aln_seq:
            if aa == '-':
                codon_aln.append('---')
            else:
                codon_aln.append(nuc_seq[nuc_idx:nuc_idx+3])
                nuc_idx += 3
        codon_aln_records.append(SeqRecord(Seq("".join(codon_aln)), id=prot_rec.id, description=""))
        
    SeqIO.write(codon_aln_records, codon_aln_fasta, "fasta")
    
    # Write to PHYLIP format required by PAML
    with open(codon_aln_phylip, 'w') as f:
        f.write(f" {len(codon_aln_records)} {len(codon_aln_records[0].seq)}\n")
        for rec in codon_aln_records:
            name = str(rec.id)[:30].ljust(30) 
            f.write(f"{name}  {str(rec.seq)}\n")
            
    print(f"反向翻译完成，输出文件：{codon_aln_phylip}")
    return codon_aln_fasta, codon_aln_phylip

codon_aln, codon_phy = align_proteins_and_back_translate(prot_fasta, clean_nuc, prefix, out_dir)

Command executed successfully: mafft --auto tutorial_output/alignments/test_gene.prot.fasta > tutorial_output/alignments/test_gene.prot.aln.fasta
反向翻译完成，输出文件：tutorial_output/alignments/test_gene.codon.aln.phy


## 3. 构建系统发育树
Codeml 的运算高度依赖基因所在的系统发育树。我们将使用快速准确的 `FastTree` 建立基于密码子比对的无根树。

In [4]:
def build_tree(codon_aln_fasta, prefix, out_dir):
    tree_file = os.path.join(out_dir, "trees", f"{prefix}.tree")
    cmd = f"FastTree -nt {shlex.quote(codon_aln_fasta)} > {shlex.quote(tree_file)}"
    run_command(cmd)
    
    unrooted_tree = os.path.join(out_dir, "trees", f"{prefix}.unrooted.tree")
    shutil.copy(tree_file, unrooted_tree)
    print(f"建树完成：{unrooted_tree}")
    
    # 打印树的内容看看
    with open(unrooted_tree, 'r') as f:
        print("树文件内容：", f.read().strip())
    
    return unrooted_tree

tree_file = build_tree(codon_aln, prefix, out_dir)

Command executed successfully: FastTree -nt tutorial_output/alignments/test_gene.codon.aln.fasta > tutorial_output/trees/test_gene.tree
建树完成：tutorial_output/trees/test_gene.unrooted.tree
树文件内容： (seq1:0.0,seq2:0.0,seq3:0.0);


## 4. 运行 PAML (Codeml) 寻找正向选择
我们使用 PAML 中的 M7 和 M8 两个位点模型 (site models) 进行对比：
- **M7 模型 (零假设, Null Model)**：假设所有密码子位点的 dN/dS (ω) < 1 或 = 1，不允许存在正向选择。
- **M8 模型 (备择假设, Alternative Model)**：允许部分位点的 ω > 1，即存在正向选择。

我们会编写对应的 `codeml.ctl` 配置文件，并分别运行这两个模型。

In [5]:
def write_codeml_ctl(ctl_file, seq_file, tree_file, out_file, model_type):
    nssites = 7 if model_type == 7 else 8
    ctl_content = f"""seqfile = {seq_file}
treefile = {tree_file}
outfile = {out_file}

noisy = 9
verbose = 1
runmode = 0
seqtype = 1
CodonFreq = 2
clock = 0
aaDist = 0
model = 0
NSsites = {nssites}
icode = 0
Mgene = 0
fix_kappa = 0
kappa = 2
fix_omega = 0
omega = 0.4
fix_alpha = 1
alpha = 0
Malpha = 0
ncatG = 10
getSE = 0
RateAncestor = 0
Small_Diff = .5e-6
cleandata = 1
fix_blength = 0
"""
    with open(ctl_file, 'w') as f:
        f.write(ctl_content)

def run_paml(aln_phy, tree_file, prefix, out_dir):
    paml_dir = os.path.join(out_dir, "paml_logs", prefix)
    os.makedirs(paml_dir, exist_ok=True)
    
    abs_aln = os.path.abspath(aln_phy)
    abs_tree = os.path.abspath(tree_file)
    abs_m7_out = os.path.abspath(os.path.join(paml_dir, "m7.out"))
    abs_m8_out = os.path.abspath(os.path.join(paml_dir, "m8.out"))

    m7_ctl = os.path.join(paml_dir, "m7.ctl")
    write_codeml_ctl(m7_ctl, abs_aln, abs_tree, abs_m7_out, 7)
    
    m8_ctl = os.path.join(paml_dir, "m8.ctl")
    write_codeml_ctl(m8_ctl, abs_aln, abs_tree, abs_m8_out, 8)
    
    curr_dir = os.getcwd()
    try:
        os.chdir(paml_dir)
        print(">>> 正在运行 PAML M7 ...")
        run_command("codeml m7.ctl", log_error=False)
        print(">>> 正在运行 PAML M8 ...")
        run_command("codeml m8.ctl", log_error=False)
    except Exception as e:
        print("Codeml 运行报错，但可能是预期内的无统计学意义数据引起的：", e)
    finally:
        os.chdir(curr_dir)
        
    return abs_m7_out, abs_m8_out

m7_out, m8_out = run_paml(codon_phy, tree_file, prefix, out_dir)

>>> 正在运行 PAML M7 ...


Command executed successfully: codeml m7.ctl
>>> 正在运行 PAML M8 ...


Command executed successfully: codeml m8.ctl


## 5. 结果分析与解释 (似然比检验 LRT)
Codeml 运行完毕后，我们需要：
1. 提取 M7 和 M8 模型的最大对数似然值 (`lnL`)。
2. 进行 **似然比检验 (Likelihood Ratio Test, LRT)**：计算统计量 `2 * (lnL_M8 - lnL_M7)`。
3. 利用卡方分布 (`df=2`) 计算 p-value。如果 `p < 0.05`，说明 M8 显著优于 M7，即**该基因受到了正向选择**。
4. 如果显著，我们从 M8 输出中提取后验概率 (BEB) > 95% 的具体氨基酸位点。

In [6]:
def parse_codeml_out(out_file):
    lnL = None
    if not os.path.exists(out_file):
        return None
    with open(out_file, 'r') as f:
        for line in f:
            if line.startswith("lnL(ntime:"):
                match = re.search(r"np:\s*\d+\):\s*([-+0-9.]+)", line)
                if match:
                    lnL = float(match.group(1))
    return lnL

def extract_beb_sites(m8_out):
    sites = []
    in_beb = False
    with open(m8_out, 'r') as f:
        for line in f.readlines():
            if "Bayes Empirical Bayes (BEB) analysis" in line:
                in_beb = True
                continue
            if in_beb and "The grid" in line:
                break
            if in_beb and "*" in line:
                parts = line.strip().split()
                if len(parts) >= 3:
                    site_pos, aa, prob_str = parts[0], parts[1], parts[2]
                    if prob_str.endswith("**"):
                        sites.append({'site': site_pos, 'aa': aa, 'prob': prob_str[:-2], 'sig': "** (>99%)"})
                    elif prob_str.endswith("*"):
                        sites.append({'site': site_pos, 'aa': aa, 'prob': prob_str[:-1], 'sig': "* (>95%)"})
    return sites

def perform_lrt_and_report(m7_out, m8_out, prefix, out_dir):
    lnL_m7 = parse_codeml_out(m7_out)
    lnL_m8 = parse_codeml_out(m8_out)
    
    if lnL_m7 is None or lnL_m8 is None:
        print("错误：无法读取似然值。Codeml 运行可能失败。")
        return
        
    lrt_stat = 2 * (lnL_m8 - lnL_m7)
    p_val = chi2.sf(lrt_stat, df=2)
    
    print(f"=== {prefix} 正向选择分析报告 ===")
    print(f"M7 (零假设) 对数似然值: {lnL_m7:.4f}")
    print(f"M8 (备择假设) 对数似然值: {lnL_m8:.4f}")
    print(f"LRT 统计量: {lrt_stat:.4f}")
    print(f"P-value: {p_val:.4e}")
    
    if p_val < 0.05:
        print("\n结论：发现显著的正向选择信号 (p < 0.05)。")
        beb_sites = extract_beb_sites(m8_out)
        if beb_sites:
            print("\n受到正向选择的特定位点 (BEB > 95%):")
            print(f"{'位点':<10}{'氨基酸':<10}{'概率':<10}{'显著性'}")
            print("-" * 40)
            for s in beb_sites:
                print(f"{s['site']:<10}{s['aa']:<10}{s['prob']:<10}{s['sig']}")
    else:
        print("\n结论：未发现显著的正向选择信号 (p >= 0.05)。")

perform_lrt_and_report(m7_out, m8_out, prefix, out_dir)

=== test_gene 正向选择分析报告 ===
M7 (零假设) 对数似然值: -35.0307
M8 (备择假设) 对数似然值: -35.0307
LRT 统计量: 0.0001
P-value: 9.9997e-01

结论：未发现显著的正向选择信号 (p >= 0.05)。
